In [1]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import os
import random
import warnings
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import KFold

warnings.filterwarnings('ignore')

# ==============================================================================
# 設定
# ==============================================================================
if os.path.exists('/kaggle/input/joai-competition-2026/train.csv'):
    INPUT_DIR = '/kaggle/input/joai-competition-2026'
elif os.path.exists('/kaggle/input/ai2026/train.csv'):
    INPUT_DIR = '/kaggle/input/ai2026'
else:
    INPUT_DIR = '.' 

TRAIN_PATH = os.path.join(INPUT_DIR, 'train.csv')
TEST_PATH = os.path.join(INPUT_DIR, 'test.csv')
SUBMISSION_PATH = 'submission_neural_ode.csv'

# 設定
MAX_LEN = 40
BATCH_SIZE = 64
N_FOLDS = 5
EPOCHS = 25  # ODEは計算が重いため少なめに
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using Device: {DEVICE}")

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)

# ==============================================================================
# 前処理 (これまでの知見を活用)
# ==============================================================================
def add_features(df):
    df_eng = df.copy()
    ignore_cols = ['id', 'sample_id', 'day_n', 'time', 'lever', 'mouse_id']
    base_cols = [c for c in df_eng.columns if c not in ignore_cols]
    
    grouped = df_eng.groupby('sample_id')
    max_day = df_eng['day_n'].max()
    gain = 1.0 + (df_eng['day_n'] / max_day * 0.5)
    
    for col in base_cols:
        df_eng[f'{col}_w'] = df_eng[col] * gain
        df_eng[f'{col}_d1'] = grouped[col].diff().fillna(0) * gain
        # ODEは微分の世界なので、積分量(cumsum)を入れると物理的に意味を持ちやすい
        df_eng[f'{col}_cum'] = grouped[col].cumsum()
        
    return df_eng

print(">>> Loading & Preprocessing...")
train = pd.read_csv(TRAIN_PATH).fillna(0)
test = pd.read_csv(TEST_PATH).fillna(0)

train = add_features(train)
test = add_features(test)

ignore_cols = ['id', 'sample_id', 'day_n', 'time', 'lever', 'mouse_id']
train_feats = [c for c in train.columns if c not in ignore_cols]
test_feats = set(test.columns)
feature_cols = [c for c in train_feats if c in test_feats]

scaler = RobustScaler()
train[feature_cols] = scaler.fit_transform(train[feature_cols])
test[feature_cols] = scaler.transform(test[feature_cols])

# ==============================================================================
# Dataset
# ==============================================================================
class BrainDataset(Dataset):
    def __init__(self, df, feature_cols, target_col=None, max_len=40):
        self.df = df
        self.sample_ids = df['sample_id'].unique()
        self.feature_cols = feature_cols
        self.target_col = target_col
        self.max_len = max_len
        self.grouped = df.groupby('sample_id')

    def __len__(self):
        return len(self.sample_ids)

    def __getitem__(self, idx):
        sample_id = self.sample_ids[idx]
        group = self.grouped.get_group(sample_id)
        x = group[self.feature_cols].values
        seq_len = len(x)
        x_pad = np.zeros((self.max_len, len(self.feature_cols)), dtype=np.float32)
        x_pad[:seq_len, :] = x
        # ODEソルバ用に時間軸情報を作成 (0.0 ~ 1.0 に正規化)
        t = np.linspace(0, 1, self.max_len, dtype=np.float32)
        
        mask = np.zeros((self.max_len,), dtype=np.float32)
        mask[:seq_len] = 1.0
        
        ret = {'x': torch.tensor(x_pad), 't': torch.tensor(t), 'mask': torch.tensor(mask), 'id': sample_id}
        
        if self.target_col:
            y = group[self.target_col].values
            y_pad = np.zeros((self.max_len,), dtype=np.float32)
            y_pad[:seq_len] = y
            ret['y'] = torch.tensor(y_pad)
        return ret

# ==============================================================================
# Neural ODE Logic (Runge-Kutta 4 Solver)
# ==============================================================================
class ODEFunc(nn.Module):
    """
    微分方程式 dy/dt = f(y, t) を定義するニューラルネット
    """
    def __init__(self, hidden_dim):
        super(ODEFunc, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(), # ODEは滑らかな関数が良いのでTanhやSoftplusが好まれる
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        
    def forward(self, t, y):
        return self.net(y)

class LatentODE(nn.Module):
    def __init__(self, input_dim, hidden_dim=128):
        super(LatentODE, self).__init__()
        
        # 1. Encoder: 入力系列全体を読んで「初期状態 z0」を決める
        # 逆向きに読む(Reverse GRU)ことで、直近の状態を重視したz0を作る
        self.encoder = nn.GRU(
            input_dim, hidden_dim, 
            batch_first=True, bidirectional=False
        )
        
        # 2. ODE Function: 状態の遷移ルール
        self.ode_func = ODEFunc(hidden_dim)
        
        # 3. Decoder: 潜在状態 z_t から予測値 lever_t への写像
        self.decoder = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.GELU(),
            nn.Linear(64, 1)
        )
        
        self.hidden_dim = hidden_dim

    def rk4_step(self, func, t, dt, y):
        """ルンゲ・クッタ法 (4次) による1ステップ積分"""
        k1 = func(t, y)
        k2 = func(t + dt/2, y + dt*k1/2)
        k3 = func(t + dt/2, y + dt*k2/2)
        k4 = func(t + dt, y + dt*k3)
        return y + dt * (k1 + 2*k2 + 2*k3 + k4) / 6

    def forward(self, x, time_steps):
        # x: [Batch, Len, InputDim]
        # time_steps: [Batch, Len] (0~1)
        
        # --- Encoder ---
        # 系列を逆順にしてGRUに通し、最後の隠れ層を取得 -> これが初期値 z(0)
        # x_flip = torch.flip(x, [1]) 
        # _, h_n = self.encoder(x_flip)
        # z0 = h_n[-1] # [Batch, Hidden]
        
        # 今回はシンプルに順方向GRUの最終状態を使う
        out, h_n = self.encoder(x)
        z0 = h_n[-1]
        
        # --- ODE Solve (Integration) ---
        # z0 からスタートして、time_steps に従って z(t) を生成
        # バッチ処理のため、全バッチ共通の dt でループを回す
        
        batch_size = x.size(0)
        seq_len = x.size(1)
        z_t = z0
        
        zs = [z0.unsqueeze(1)] # t=0 の状態
        
        # 簡易化: 時間ステップは等間隔(dt)と仮定
        dt = 1.0 / (seq_len - 1)
        
        for i in range(seq_len - 1):
            t_curr = i * dt
            # RK4で次の状態へ積分
            z_t = self.rk4_step(self.ode_func, t_curr, dt, z_t)
            zs.append(z_t.unsqueeze(1))
            
        # [Batch, Len, Hidden]
        z_sequence = torch.cat(zs, dim=1)
        
        # --- Decoder ---
        # 軌道 z(t) から値を予測
        pred = self.decoder(z_sequence)
        
        return pred.squeeze(-1)

# ==============================================================================
# Training Loop
# ==============================================================================
final_test_preds = {} 
test_counts = {}

unique_samples = train['sample_id'].unique()
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
criterion = nn.MSELoss(reduction='none')

print(f"\n>>> Starting Neural ODE Training ({N_FOLDS} folds)...")
print("  Note: ODE solver is computationally expensive. Training may be slower.")

test_ds = BrainDataset(test, feature_cols, None, MAX_LEN)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

for fold, (train_idx, val_idx) in enumerate(kf.split(unique_samples)):
    print(f"\n=== Fold {fold+1}/{N_FOLDS} ===")
    
    train_ids, val_ids = unique_samples[train_idx], unique_samples[val_idx]
    train_ds = BrainDataset(train[train['sample_id'].isin(train_ids)], feature_cols, 'lever', MAX_LEN)
    val_ds = BrainDataset(train[train['sample_id'].isin(val_ids)], feature_cols, 'lever', MAX_LEN)
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    
    model = LatentODE(
        input_dim=len(feature_cols),
        hidden_dim=128
    ).to(DEVICE)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    
    best_loss = float('inf')
    best_model_path = f"best_ode_fold{fold}.pth"
    
    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        for batch in train_loader:
            x = batch['x'].to(DEVICE)
            t = batch['t'].to(DEVICE)
            y = batch['y'].to(DEVICE)
            m = batch['mask'].to(DEVICE)
            
            optimizer.zero_grad()
            preds = model(x, t)
            
            # Loss計算
            loss = (criterion(preds, y) * m).sum() / m.sum()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item()
        
        scheduler.step()
        
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                x = batch['x'].to(DEVICE)
                t = batch['t'].to(DEVICE)
                y = batch['y'].to(DEVICE)
                m = batch['mask'].to(DEVICE)
                
                preds = model(x, t)
                loss = (criterion(preds, y) * m).sum() / m.sum()
                val_loss += loss.item()
        
        avg_val = val_loss / len(val_loader)
        if avg_val < best_loss:
            best_loss = avg_val
            torch.save(model.state_dict(), best_model_path)
            
        if (epoch+1) % 5 == 0:
            print(f"  Ep {epoch+1:2d} | Train: {train_loss/len(train_loader):.4f} | Val: {avg_val:.4f}")

    print(f"  >> Best Val: {best_loss:.5f}")

    # Inference
    try:
        model.load_state_dict(torch.load(best_model_path))
        model.eval()
        with torch.no_grad():
            for batch in test_loader:
                x = batch['x'].to(DEVICE)
                t = batch['t'].to(DEVICE)
                mask = batch['mask'].to(DEVICE)
                s_ids = batch['id']
                
                preds = model(x, t).cpu().numpy()
                mask = mask.cpu().numpy()
                
                for i, s_id in enumerate(s_ids):
                    v_len = int(mask[i].sum())
                    for time_idx, val in enumerate(preds[i, :v_len]):
                        k = f"{s_id}_{time_idx}"
                        final_test_preds[k] = final_test_preds.get(k, 0) + val
                        test_counts[k] = test_counts.get(k, 0) + 1
        
        # 中間保存
        temp_results = [{'id': k, 'lever': max(0, v / test_counts[k])} for k, v in final_test_preds.items()]
        pd.DataFrame(temp_results).to_csv(f'temp_submission_ode_fold{fold+1}.csv', index=False)
        
    except Exception as e:
        print(f"  Error in Inference: {e}")

# ==============================================================================
# Final Submission
# ==============================================================================
if len(final_test_preds) > 0:
    results = [{'id': k, 'lever': max(0, v / test_counts[k])} for k, v in final_test_preds.items()]
    df_sub = pd.DataFrame(results)
    df_sub.to_csv(SUBMISSION_PATH, index=False)
    print(f"\nSUCCESS: {SUBMISSION_PATH} created.")
else:
    print("ERROR: No predictions.")

Using Device: cuda
>>> Loading & Preprocessing...

>>> Starting Neural ODE Training (5 folds)...
  Note: ODE solver is computationally expensive. Training may be slower.

=== Fold 1/5 ===
  Ep  5 | Train: 0.9908 | Val: 1.1481
  Ep 10 | Train: 0.8025 | Val: 1.0748
  Ep 15 | Train: 0.6621 | Val: 1.1156
  Ep 20 | Train: 0.5556 | Val: 1.1051
  Ep 25 | Train: 0.5225 | Val: 1.0986
  >> Best Val: 1.05364

=== Fold 2/5 ===
  Ep  5 | Train: 1.0068 | Val: 1.0353
  Ep 10 | Train: 0.8120 | Val: 1.0457
  Ep 15 | Train: 0.6557 | Val: 1.0369
  Ep 20 | Train: 0.5577 | Val: 1.0735
  Ep 25 | Train: 0.5216 | Val: 1.0726
  >> Best Val: 0.98729

=== Fold 3/5 ===
  Ep  5 | Train: 1.0146 | Val: 1.0738
  Ep 10 | Train: 0.8124 | Val: 1.0459
  Ep 15 | Train: 0.6544 | Val: 1.0480
  Ep 20 | Train: 0.5538 | Val: 1.1031
  Ep 25 | Train: 0.5161 | Val: 1.1112
  >> Best Val: 1.03582

=== Fold 4/5 ===
  Ep  5 | Train: 0.9918 | Val: 1.0714
  Ep 10 | Train: 0.8066 | Val: 1.0310
  Ep 15 | Train: 0.6553 | Val: 1.0307
  Ep 